In [2]:
# import
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, regularizers, models


In [9]:
#preprocessing
datapath = "../../../../desktop/quant/hist/aaplIntra.csv"
df = pd.read_csv(datapath)


In [10]:
df

,Dates,Open,Close,High,Low,Volume,Number Ticks
0,7/1/25 9:30,206.665,206.915,207.08,206.600,1035492,1433
1,7/1/25 9:30,206.910,206.710,206.92,206.500,119487,721
2,7/1/25 9:30,206.730,206.810,206.95,206.695,95679,603
3,7/1/25 9:30,206.840,207.200,207.22,206.790,164543,948
4,7/1/25 9:30,207.200,207.115,207.24,206.980,123276,626
...,...,...,...,...,...,...,...
281104,##########,273.900,273.670,274.60,273.470,95766657,2970
281105,##########,273.670,273.670,273.67,273.670,0,1
281106,12/19/25 15:59,273.690,273.900,273.91,273.600,556667,1933
281107,12/19/25 15:59,273.900,273.670,274.60,273.470,95766657,2970


In [ ]:
feature_cols = ["Open", "High", "Low", "Close", "Volume", "Number Ticks"]
X_all = df[feature_cols].values.astype("float32")
# next bar price label from close
close = df["Close"].values.astype("float32")
y_all = np.roll(close, -1)
# drop last bar
X_all = X_all[:-1]
y_all = y_all[:-1]
# turn into sequences of length T
T = 600
Xs = []
ys = []
for i in range(len(X_all) - T + 1):
    Xs.append(X_all[i:i+T])
    ys.append(y_all[i+T-1])
Xs = np.stack(Xs)        # shape (N, T, F)
ys = np.array(ys)        # shape (N,)
# random train validation split 80 / 20
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(Xs, ys, test_size=0.2, random_state=42)

In [12]:
timesteps = X_train.shape[1]
features = X_train.shape[2]

model = tf.keras.Sequential([
    tf.keras.layers.LSTM(128, return_sequences=True, input_shape=(timesteps, features), dropout=0.2),
    tf.keras.layers.LSTM(64, return_sequences=False, dropout=0.2),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(1)  #linear activation for regression
])

model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

model.summary()
model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=1,
    batch_size=32
)


/opt/miniconda3/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 600, 128)       │        69,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 120,641 (471.25 KB)

 Trainable params: 120,641 (471.25 KB)

 Non-trainable params: 0 (0.00 B)

  16/7013 ━━━━━━━━━━━━━━━━━━━━ 56:42 486ms/step - loss: 59861.1914 - mae: 243.3781

KeyboardInterrupt: 

In [ ]:
model.save('models/lstm.keras')